# 02 - Feature Analysis

Analyze the extracted FS-W-PA-14 features stored in HDF5 files.

**Steps:**
1. Load and inspect HDF5 feature files
2. Feature distribution analysis per sleep stage
3. Feature correlation matrix
4. Feature importance (discriminative power for 5-class staging)
5. Window/mask quality analysis

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
%matplotlib inline

PROJECT_DIR = Path("/data/priorj/sleep_stage_classification")
FEATURE_DIR = PROJECT_DIR / "datasets" / "store" / "fs-w-pa-14-60"

FEATURE_NAMES = [
    "hr_bpm", "hrv_td_mean_nn", "hrv_td_sd_nn", "hrv_td_median_nn",
    "hrv_fd_lfhf_ratio", "spo2_mu", "spo2_std", "spo2_med",
    "mov_mu", "mov_std", "mov_med", "rsp_bpm", "spo2_qos", "hrv_qos"
]

STAGE_NAMES = ["WAKE", "N1", "N2", "N3", "REM"]
CLASS_MAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3, 5: 4}  # 5-class mapping

## 1. Load and Inspect Feature Files

In [ ]:
# Discover all HDF5 files
h5_files = sorted(FEATURE_DIR.rglob("*.h5"))
print(f"Found {len(h5_files)} feature files")

# Group by dataset
for dataset in ["mesa", "stages"]:
    ds_files = [f for f in h5_files if dataset in str(f)]
    print(f"  {dataset}: {len(ds_files)} subjects")

# Inspect first file
if h5_files:
    with h5py.File(str(h5_files[0]), "r") as f:
        print(f"\nSample file: {h5_files[0].name}")
        print(f"  Datasets: {list(f.keys())}")
        for key in f.keys():
            print(f"  {key}: shape={f[key].shape}, dtype={f[key].dtype}")

In [ ]:
# Load features from a sample of subjects for analysis
N_SAMPLE = min(100, len(h5_files))
rng = np.random.RandomState(42)
sample_files = rng.choice(h5_files, N_SAMPLE, replace=False)

all_features = []
all_labels = []
all_masks = []

for fpath in sample_files:
    with h5py.File(str(fpath), "r") as f:
        features = f["features"][:]
        labels = f["stage_labels"][:]
        mask = f["mask"][:]
        
        # Apply mask
        valid = mask == 1
        all_features.append(features[valid])
        all_labels.append(labels[valid])
        all_masks.append(mask)

features_arr = np.concatenate(all_features)
labels_arr = np.concatenate(all_labels)

# Apply class map
mapped_labels = np.array([CLASS_MAP.get(l, -1) for l in labels_arr])
valid_mask = mapped_labels >= 0
features_arr = features_arr[valid_mask]
mapped_labels = mapped_labels[valid_mask]

print(f"Loaded {len(features_arr)} valid windows from {N_SAMPLE} subjects")
print(f"Feature shape: {features_arr.shape}")

## 2. Feature Distributions by Sleep Stage

In [ ]:
# Create DataFrame for easier plotting
feat_df = pd.DataFrame(features_arr, columns=FEATURE_NAMES[:features_arr.shape[1]])
feat_df["stage"] = [STAGE_NAMES[l] for l in mapped_labels]

# Box plots for each feature grouped by sleep stage
n_features = min(len(FEATURE_NAMES), features_arr.shape[1])
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i in range(min(n_features, 16)):
    feat_name = FEATURE_NAMES[i]
    # Clip outliers for visualization
    data = feat_df[[feat_name, "stage"]].copy()
    q01, q99 = data[feat_name].quantile([0.01, 0.99])
    data = data[(data[feat_name] >= q01) & (data[feat_name] <= q99)]
    
    sns.boxplot(data=data, x="stage", y=feat_name, ax=axes[i],
                order=STAGE_NAMES, palette="Set2")
    axes[i].set_title(feat_name)
    axes[i].set_xlabel("")

# Hide unused subplots
for i in range(n_features, len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Feature Distributions by Sleep Stage", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Correlation Matrix

In [ ]:
# Correlation matrix of features
corr = feat_df[FEATURE_NAMES[:n_features]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

# Identify highly correlated feature pairs
print("Highly correlated feature pairs (|r| > 0.8):")
for i in range(n_features):
    for j in range(i+1, n_features):
        if abs(corr.iloc[i, j]) > 0.8:
            print(f"  {FEATURE_NAMES[i]} <-> {FEATURE_NAMES[j]}: r={corr.iloc[i,j]:.3f}")

## 4. Feature Discriminative Power

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Quick feature importance using Random Forest
# Subsample for speed
n_sub = min(50000, len(features_arr))
idx = rng.choice(len(features_arr), n_sub, replace=False)
X = features_arr[idx]
y = mapped_labels[idx]

# Handle NaN
nan_mask = ~np.any(np.isnan(X), axis=1)
X = X[nan_mask]
y = y[nan_mask]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_scaled, y)

importances = rf.feature_importances_
feat_importance = pd.DataFrame({
    "feature": FEATURE_NAMES[:X.shape[1]],
    "importance": importances
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_importance["feature"], feat_importance["importance"], color="#3498db")
ax.set_title("Feature Importance (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

print(f"\nRandom Forest accuracy on subsample: {rf.score(X_scaled, y):.3f}")

## 5. Mask / Quality Analysis

In [ ]:
# Analyze mask validity across subjects
mask_stats = []
for fpath in sample_files:
    with h5py.File(str(fpath), "r") as f:
        mask = f["mask"][:]
        mask_stats.append({
            "file": fpath.stem,
            "total_windows": len(mask),
            "valid_windows": int(np.sum(mask)),
            "valid_pct": np.mean(mask) * 100
        })

mask_df = pd.DataFrame(mask_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(mask_df["valid_pct"], bins=30, color="#2ecc71", edgecolor="white")
axes[0].set_title("Mask Validity (% valid windows per subject)")
axes[0].set_xlabel("% Valid")
axes[0].set_ylabel("Count")

axes[1].hist(mask_df["total_windows"], bins=30, color="#3498db", edgecolor="white")
axes[1].set_title("Total Windows per Subject")
axes[1].set_xlabel("Number of windows")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Mask validity statistics:")
print(mask_df["valid_pct"].describe())

## Summary

Key findings:
- Which features have the most discriminative power for sleep staging
- Feature correlations (identify redundancy)
- Data quality (mask validity distribution)
- Stage-specific feature patterns (e.g., HRV during REM vs NREM)

**Next steps**: Train the TCN model (Phase 3) and analyze results in notebook 03.